# 2026-03 Basics of docling

- Example and fake data used for this study:
    - Microsoft Azure invoices: https://www.microsoft.com/en-us/download/details.aspx?id=38805
    - Kaggle receipts (2024-EN): https://www.kaggle.com/datasets/jenswalter/receipts

**Note**: I've found a bunch of errors when trying to install pymupdf on macOs arm based systems. At the moment of writing this doc, there is no conscent on the web of the right way to do it, to fix it I've used a very ugly command `uv pip install pymupdf`.


---

### Setup
Importing all necessary libs and geting a list with all the example data

In [1]:
# Python imports
import os, sys
from pathlib import Path

# Data handling imports
import pandas as pd
import numpy as np
from tqdm import tqdm

# --> Main import: pymupdf
import pymupdf

In [2]:
# Path to all the example files
path = Path('./data/')

# Get all files
files_list = [f'./data/{entry.name}' for entry in path.rglob('*') if entry.is_file()]

# Sort the list
files_list = sorted(files_list)


### Helper functions

In [3]:
def helper_doc_report(doc: Document):
    """
    Helper function to give basic info about a
    imported document via pymupdf.
    """

    # Get metadata from the pdf
    doc_metadata = doc.metadata
    
    # Print of the imported doc
    
    print("="*80)
    print(f"Document: {doc}")
    print(f"Format: {doc_metadata.get('format')} | Creator: {doc_metadata.get('creator')}")
    print("="*80)
    print(f"Number of pages: {doc.page_count}")
    print("="*80)
    print(f"Encryptions: {doc_metadata.get('encryption')}")
    print("")    

In [4]:
def helper_render_page(doc, page_number: int = 0, zoom: float = 1.0):
    """
    Helper function to render a PDF page as a pixmap (image).
    Returns the rendered pymupdf.Pixmap object.
    """
    page = doc.load_page(page_number)

    # Create transformation matrix for zoom
    mat = pymupdf.Matrix(zoom, zoom)

    # Render the page to a pixmap
    pix = page.get_pixmap(matrix=mat)

    print("="*80)
    print(f"Page {page_number} rendered")
    print(f"Size: {pix.width}x{pix.height} pixels | Color space: {pix.colorspace.name}")
    print(f"Zoom factor: {zoom}x")
    print("="*80)

    return pix

In [5]:
def helper_extract_images(doc, page_number: int = 0):
    """
    Helper function to extract all embedded images from a PDF page.
    Returns a list of image data dicts (keys: ext, width, height, image bytes).
    """
    page = doc.load_page(page_number)

    # Get list of image references on the page (full=True for complete metadata)
    image_list = page.get_images(full=True)

    print("="*80)
    print(f"Page {page_number} — {len(image_list)} embedded image(s) found")
    print("="*80)

    images = []
    for i, img_ref in enumerate(image_list):
        # img_ref tuple: (xref, smask, width, height, bpc, colorspace, ...)
        xref = img_ref[0]
        img_data = doc.extract_image(xref)
        print(f"  Image {i}: xref={xref} | ext={img_data['ext']} | size={img_data['width']}x{img_data['height']}")
        images.append(img_data)

    print("")
    return images

### Basic testing
Just simple 'out of the box' examples and pipelines

In [7]:
# Using the receit from cheesecakefactory
doc = pymupdf.open(files_list[1])

helper_doc_report(doc)


Document: Document('./data/example-receipts-kaggle-cheesecakefactory.pdf')
Format: PDF 1.7 | Creator: PFU ScanSnap Cloud #iX500
Number of pages: 1
Encryptions: None



In [8]:
doc.page_count

1

### Text Extraction
Different modes of extracting text from a PDF page using `page.get_text(mode)`

In [26]:
# Open the invoice for text extraction examples
doc_invoice = pymupdf.open(files_list[1])
page = doc_invoice.load_page(0)

# Mode "text" — simplest plain text, reading order top-to-bottom
plain_text = page.get_text("text")

print("=== Plain text extraction (mode='text') ===")
print(plain_text[100:200])

=== Plain text extraction (mode='text') ===
/24
1
HH Nac
& JaCks
Dr
1
Louisi(1na Chicken Pasta
1
Nac
&
%lacks
Afi､ican Amb
1
Fresh Sti"dwberl y 


In [27]:
# Mode "blocks" — text grouped into rectangular blocks with bounding boxes
# Each block: (x0, y0, x1, y1, "text", block_no, block_type)
# block_type: 0 = text block, 1 = image block
blocks = page.get_text("blocks")

print(f"=== Blocks extraction (mode='blocks'): {len(blocks)} block(s) found ===\n")
for block in blocks[:6]:
    x0, y0, x1, y1, text, block_no, block_type = block
    print(f"Block {block_no} (type={block_type}) | bbox=[{x0:.1f}, {y0:.1f}, {x1:.1f}, {y1:.1f}]")
    print(f"  {text.strip()[:80]!r}")
    print()

=== Blocks extraction (mode='blocks'): 34 block(s) found ===

Block 0 (type=0) | bbox=[61.4, 54.5, 160.4, 73.7]
  '||上CI1t[5上CA|<[\n| ACIORY\nSEA｢｢LE'

Block 1 (type=0) | bbox=[13.4, 89.0, 141.5, 120.7]
  'O235\nTABLE 203\n#Party\nDALTON\nI_\nSVrCk:\n5\n17152\nBAR\nDININ{:i'

Block 2 (type=0) | bbox=[147.6, 89.3, 184.6, 109.9]
  '1\n05/?0/24'

Block 3 (type=0) | bbox=[14.2, 147.8, 102.8, 155.8]
  '1\nHH Nac\n& JaCks\nDr'

Block 4 (type=0) | bbox=[14.2, 159.4, 131.2, 167.5]
  '1\nLouisi(1na Chicken Pasta'

Block 5 (type=0) | bbox=[14.2, 170.9, 131.6, 179.3]
  '1\nNac\n&\n%lacks\nAfi､ican Amb'



In [28]:
# Mode "words" — individual words with their bounding boxes and position indices
# Each word: (x0, y0, x1, y1, "word", block_no, line_no, word_no)
words = page.get_text("words")

print(f"=== Words extraction (mode='words'): {len(words)} word(s) found ===\n")
for word in words[:12]:
    x0, y0, x1, y1, text, block_no, line_no, word_no = word
    print(f"  {text!r:20s} @ [{x0:.1f}, {y0:.1f}, {x1:.1f}, {y1:.1f}] (b={block_no}, l={line_no}, w={word_no})")

=== Words extraction (mode='words'): 133 word(s) found ===

  '||上CI1t[5上CA|<['    @ [61.4, 54.5, 121.4, 61.9] (b=0, l=0, w=0)
  '|'                  @ [128.9, 54.5, 129.9, 61.9] (b=0, l=1, w=0)
  'ACIORY'             @ [133.4, 54.5, 160.4, 61.9] (b=0, l=1, w=1)
  'SEA｢｢LE'            @ [95.0, 65.8, 127.0, 73.7] (b=0, l=2, w=0)
  'O235'               @ [13.4, 89.0, 49.4, 98.4] (b=1, l=0, w=0)
  'TABLE'              @ [61.4, 89.0, 83.4, 98.4] (b=1, l=1, w=0)
  '203'                @ [90.2, 89.0, 103.2, 98.4] (b=1, l=1, w=1)
  '#Party'             @ [114.0, 89.0, 141.0, 98.4] (b=1, l=2, w=0)
  'DALTON'             @ [13.4, 100.6, 40.4, 109.0] (b=1, l=3, w=0)
  'I_'                 @ [47.5, 100.6, 50.5, 109.0] (b=1, l=4, w=0)
  'SVrCk:'             @ [70.8, 100.6, 96.8, 109.0] (b=1, l=5, w=0)
  '5'                  @ [109.4, 100.6, 112.4, 109.0] (b=1, l=6, w=0)


In [29]:
# Mode "dict" — full structured extraction: blocks → lines → spans, with font info
text_dict = page.get_text("dict")

print(f"=== Dict extraction (mode='dict') ===")
print(f"Page size: {text_dict['width']:.0f} x {text_dict['height']:.0f} pts")
print(f"Number of blocks: {len(text_dict['blocks'])}\n")

# Dig into the first text block
for block in text_dict["blocks"][:2]:
    if block.get("type") == 0:  # text block
        print(f"Block #{block['number']} | bbox={[round(v,1) for v in block['bbox']]}")
        for line in block.get("lines", []):
            for span in line.get("spans", []):
                print(f"  font={span['font']!r:30s} size={span['size']:.1f}  text={span['text']!r}")
        print()

=== Dict extraction (mode='dict') ===
Page size: 224 x 668 pts
Number of blocks: 35

Block #1 | bbox=[61.4, 54.5, 160.4, 73.7]
  font='GlyphLessFont'                size=7.7  text='||上CI1t[5上CA|<['
  font='GlyphLessFont'                size=3.9  text='|'
  font='GlyphLessFont'                size=8.2  text=' '
  font='GlyphLessFont'                size=8.2  text='ACIORY'
  font='GlyphLessFont'                size=8.5  text='SEA｢｢LE'



### Page Rendering
Rendering PDF pages as raster images using `page.get_pixmap()`

In [30]:
# Render the receipt page at default resolution (72 DPI, zoom=1.0)
doc_receipt = pymupdf.open(files_list[1])

pix = helper_render_page(doc_receipt, page_number=0, zoom=1.0)

# Save the rendered page as PNG
output_path = "./page_render_default.png"
pix.save(output_path)
print(f"Saved to: {output_path}")

Page 0 rendered
Size: 225x669 pixels | Color space: DeviceRGB
Zoom factor: 1.0x
Saved to: ./page_render_default.png


In [31]:
# Render at higher resolution: zoom=2.0 → 144 DPI, zoom=3.0 → 216 DPI
pix_hd = helper_render_page(doc_receipt, page_number=0, zoom=2.0)

output_path_hd = "./page_render_2x.png"
pix_hd.save(output_path_hd)

print(f"Saved HD render to: {output_path_hd}")
print(f"Page size (pts):     {doc_receipt[0].rect.width:.0f} x {doc_receipt[0].rect.height:.0f}")
print(f"Rendered size (px):  {pix_hd.width} x {pix_hd.height}")

Page 0 rendered
Size: 449x1337 pixels | Color space: DeviceRGB
Zoom factor: 2.0x
Saved HD render to: ./page_render_2x.png
Page size (pts):     224 x 668
Rendered size (px):  449 x 1337


### Image Extraction
Extracting embedded images from PDF pages using `page.get_images()` and `doc.extract_image(xref)`

In [32]:
# Scan all pages in the invoice for embedded images
doc_invoice = pymupdf.open(files_list[0])

print("=== Scanning all pages for embedded images ===\n")
for page_number in range(doc_invoice.page_count):
    page = doc_invoice.load_page(page_number)
    # get_images(full=True) returns: (xref, smask, width, height, bpc, colorspace, ...)
    image_list = page.get_images(full=True)
    print(f"Page {page_number}: {len(image_list)} image(s)")
    for img in image_list:
        print(f"  xref={img[0]} | size={img[2]}x{img[3]} px | colorspace={img[5]!r}")

=== Scanning all pages for embedded images ===

Page 0: 1 image(s)
  xref=8 | size=1250x963 px | colorspace='DeviceRGB'
Page 1: 2 image(s)
  xref=11 | size=1251x213 px | colorspace='DeviceRGB'
  xref=12 | size=1251x724 px | colorspace='DeviceRGB'
Page 2: 1 image(s)
  xref=15 | size=1249x962 px | colorspace='DeviceRGB'


In [33]:
# Extract images from page 0 using the helper and save the first one
extracted = helper_extract_images(doc_invoice, page_number=0)

if extracted:
    first_img = extracted[0]
    img_path = f"./extracted_image_p0.{first_img['ext']}"
    with open(img_path, "wb") as f:
        f.write(first_img["image"])
    print(f"Saved first extracted image to: {img_path}")
else:
    print("No embedded images found on page 0")

Page 0 — 1 embedded image(s) found
  Image 0: xref=8 | ext=png | size=1250x963

Saved first extracted image to: ./extracted_image_p0.png


### Table Detection
Finding and extracting tables from PDF pages using `page.find_tables()`

In [34]:
# Table detection on the invoice (more likely to contain structured tables)
doc_invoice = pymupdf.open(files_list[0])
page = doc_invoice.load_page(0)

# find_tables() returns a TableFinder object with a .tables list
tables = page.find_tables()

print(f"=== Table detection: {len(tables.tables)} table(s) found ===\n")
for i, table in enumerate(tables.tables):
    print(f"Table {i}:")
    print(f"  Bounding box: {[round(v, 1) for v in table.bbox]}")
    print(f"  Rows: {table.row_count} | Cols: {table.col_count}")
    print()

=== Table detection: 0 table(s) found ===



In [35]:
# Extract table data as rows and load into a DataFrame
for i, table in enumerate(tables.tables):
    print(f"=== Table {i} ===\n")

    # extract() returns a list of lists (rows of cell strings)
    table_data = table.extract()

    for row_idx, row in enumerate(table_data[:5]):
        print(f"  Row {row_idx}: {row}")
    if len(table_data) > 5:
        print(f"  ... ({len(table_data) - 5} more rows)")

    print()

    # Load into DataFrame — use first row as header if the table has an external header
    if table_data and table.header.external:
        df = pd.DataFrame(table_data[1:], columns=table_data[0])
    else:
        df = pd.DataFrame(table_data)

    display(df.head())

### Text Search
Searching for specific text within a PDF page using `page.search_for()`

In [36]:
# search_for() finds all occurrences of a string and returns their bounding boxes (Rect objects)
doc_receipt = pymupdf.open(files_list[1])
page = doc_receipt.load_page(0)

# Quick look at what text is on the page
plain_text = page.get_text("text")
print("=== Page text preview ===")
print(plain_text[:300])
print()

# Search for a term — case-insensitive by default
search_term = "Cheesecake"
hits = page.search_for(search_term)

print(f"=== Searching for: {search_term!r} ===")
print(f"Found {len(hits)} occurrence(s)\n")
for j, rect in enumerate(hits):
    print(f"  Hit {j}: x0={rect.x0:.1f}, y0={rect.y0:.1f}, x1={rect.x1:.1f}, y1={rect.y1:.1f}")

=== Page text preview ===
||上CI1t[5上CA|<[
| ACIORY
SEA｢｢LE
O235
TABLE 203
#Party
DALTON
I_
SVrCk:
5
17152
BAR
DININ{:i
1
05/?0/24
1
HH Nac
& JaCks
Dr
1
Louisi(1na Chicken Pasta
1
Nac
&
%lacks
Afi､ican Amb
1
Fresh Sti"dwberl y CC
5．50
25．95
9,00
12.95
Sub Total
:
53.40
TaX
:
5，53
05/20
18
:
40~I~O~TAI_
:
58.93
GI-atuity
Nut
l

=== Searching for: 'Cheesecake' ===
Found 1 occurrence(s)

  Hit 0: x0=60.5, y0=628.3, x1=105.5, y1=636.7
